# Convert Center Points and Calibration Coordinates to .csv

Script exports one .csv for shark center points and one .csv for calibration end points, both of which are stored in the same location as the .xml file denoting the original annotations (exported from CVAT as CVAT for Images 1.1, not COCO). Script assumes that EXIF data contains relevant information, and filenames are formatted accordingly (e.g. 07202023PANB0101.JPG).

The centerpoints.csv file contains columns necessary for parsing in the DataLoader() (center_x, center_y).

In [17]:
import os
import csv
import xml.etree.ElementTree as ET
from collections import defaultdict
import subprocess
import json
import re
from pathlib import Path
import pandas as pd
import numpy as np

In [18]:
os.chdir("/Volumes/JWS_2025/cv4e-bodycondition/sharkbody_seg/notebooks") 
notebook_path = Path.cwd()
project_root = notebook_path.parents[0] 
dataset_root = project_root / 'dataset'
exiftool_path = "/opt/homebrew/bin/exiftool"

# path to your XML file; export points annotations as 'CVAT for Images 1.1'
xml_file = dataset_root / 'metadata/centerpoints/centerpoints.xml'

# path to metadata - calibration object true lengths
calib_len_file = dataset_root / 'metadata/calibration_lengths.csv'
calib_len = pd.read_csv(calib_len_file)

# path to all your images
img_dir = dataset_root / 'images/'

# EXIF Extraction

In [19]:
def convert_decimaldeg(dms_str):
    """convert a gps dms string to decimal degrees"""
    match = re.match(r"(\d+)\s+deg\s+(\d+)'?\s+([\d.]+)\"?\s*([NSEW])", dms_str)
    degrees, minutes, seconds, direction = match.groups()
    decimal = float(degrees) + float(minutes) / 60 + float(seconds) / 3600
    if direction in ['S', 'W']:
        decimal *= -1
    return decimal

In [20]:
def parse_filename(filename):
    """pulls relevant metadata out of the filename (reqs UAS image filenames in standardized format)"""
    base = filename.split('.')[0]
    date = base[:8]
    aircraft = base[8:10]
    site = base[10:12]
    flight = base[12:14]
    image_num = base[14:16]
    return date, aircraft, site, flight, image_num

# Add new centerpoint labels from LabelBox 
Labelbox exports annotations as .ndjson files, requires conversion and merging to original .xml data before splitting

In [21]:
# Input and output paths
ndjson_file = dataset_root / 'metadata/centerpoints/additional_labels/centerpoints_11062025.ndjson' # path to your new .ndjson annotations
merged_xml_file = dataset_root / 'metadata/centerpoints/merged_centerpoints.xml'

# Load the existing XML
tree = ET.parse(xml_file) # original xml data
root = tree.getroot()

# Find last image ID (for numbering new ones)
existing_ids = [int(img.get("id")) for img in root.findall("image")]
next_id = max(existing_ids) + 1 if existing_ids else 0

# Read NDJSON annotations
with open(ndjson_file, "r") as f:
    lines = [json.loads(line.strip()) for line in f if line.strip()]

for i, entry in enumerate(lines):
    data = entry["data_row"]
    media = entry["media_attributes"]
    projects = entry["projects"]
    project_data = next(iter(projects.values()))
    labels = project_data["labels"]

    # Create new <image> block
    img_elem = ET.SubElement(
        root, "image",id=str(next_id + i),name=data["external_id"],width=str(media["width"]),
        height=str(media["height"]))

    # Collect points by label
    points_by_label = defaultdict(list)
    for label in labels:
        for obj in label["annotations"]["objects"]:
            if "point" in obj:
                x = round(obj["point"]["x"], 2)
                y = round(obj["point"]["y"], 2)
                points_by_label[obj["name"]].append(f"{x},{y}")

    # Create single <points> per label with all points joined by ;
    for label_name, point_list in points_by_label.items():
        points_str = ";".join(point_list)
        ET.SubElement(img_elem,"points",label=label_name,source="manual",
            occluded="0",points=points_str,z_order="0")
        
# Save merged XML
ET.indent(tree, space="  ", level=0)
tree.write(merged_xml_file, encoding="utf-8", xml_declaration=True)

print(f"✅ Merged NDJSON annotations into {merged_xml_file}")

✅ Merged NDJSON annotations into /Volumes/JWS_2025/cv4e-bodycondition/sharkbody_seg/dataset/metadata/centerpoints/merged_centerpoints.xml


# Compile exif data and create outputs
creates centerpoint_data and calibrationpoint_data with EXIF data embedded

In [22]:
# compile exif data and create centerpoint_data and calibrationpoints_data with EXIF data

tree = ET.parse(merged_xml_file)
root = tree.getroot()
xml_dir = os.path.dirname(merged_xml_file) # merged .xml data

centerpoint_data = []
calibrationpoints_data = []

# iterate through all images
for image in root.findall('image'):
    image_name = image.get('name')
    img_path = os.path.join(img_dir, image_name) 

    # DEBUGGING
    print(f"Image: {image_name}")

    # default EXIF values
    relative_altitude, gimbal_pitch, date_time = None, None, None
    gps_lon, gps_lat = None, None
    image_width, image_height = None, None

    # extract EXIF metadata
    try:
        result = subprocess.run([exiftool_path, "-j", img_path], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
        metadata_list = json.loads(result.stdout)

        if metadata_list:
            metadata = metadata_list[0]
            relative_altitude = float(metadata.get('RelativeAltitude', 'nan'))
            gimbal_pitch = float(metadata.get('GimbalPitchDegree', 'nan'))
            date_time = metadata.get('DateTimeOriginal')
            image_width = metadata.get('ImageWidth')
            image_height = metadata.get('ImageHeight')
            gps_lon = convert_decimaldeg(metadata.get('GPSLongitude'))
            gps_lat = convert_decimaldeg(metadata.get('GPSLatitude'))
            
        else:
            print(f"No EXIF data found for {image_name}")

    except Exception as e:
        print(f"Error reading EXIF from {image_name}: {e}")

    # extract metadata from filename
    date_str, aircraft, site, flight, image_num = parse_filename(image_name)

    # extract center point coordinates
    for points in image.findall('points'):
        label = points.get('label')
        points_str = points.get('points')

        if not points_str: 
            print(f"Missing center points in {image_name}")
            continue # skip empty points 

        coordinates = points_str.split(';')

        if label == "shark center point":
            x_str, y_str = coordinates[0].split(',')
            center_x = float(x_str.strip())
            center_y = float(y_str.strip())

            centerpoint_data.append(
                [image_name, label, center_y, center_x, relative_altitude, gimbal_pitch,
                     date_time, gps_lon, gps_lat, image_width, image_height, aircraft, site, flight])

        elif label.lower().startswith("calibration"):
            (x1_str, y1_str) = coordinates[0].split(',')
            (x2_str, y2_str) = coordinates[1].split(',')

            endpoint1_x, endpoint1_y = float(x1_str.strip()), float(y1_str.strip())
            endpoint2_x, endpoint2_y = float(x2_str.strip()), float(y2_str.strip())

            calibrationpoints_data.append(
                [image_name, label, relative_altitude, gimbal_pitch, date_time, gps_lon, gps_lat, 
                 image_width, image_height, endpoint1_x, endpoint1_y, endpoint2_x, endpoint2_y, aircraft, site, flight])

Image: 01302025PAAN0101.JPG
Image: 01302025PAAN0101C.JPG
Image: 01302025PAAN0102.JPG
Image: 01302025PAAN0102C.JPG
Image: 01302025PAAN0103.JPG
Image: 01302025PAAN0103C.JPG
Image: 01302025PAAN0104.JPG
Image: 01302025PAAN0104C.JPG
Image: 01302025PAAN0105.JPG
Image: 01302025PAAN0105C.JPG
Image: 01302025PAAN0106.JPG
Image: 01302025PAAN0106C.JPG
Image: 01302025PAAN0107.JPG
Image: 01302025PAAN0107C.JPG
Image: 01302025PAAN0108.JPG
Image: 01302025PAAN0109.JPG
Image: 01302025PAAN0110.JPG
Image: 01302025PAAN0111.JPG
Image: 01302025PAAN0112.JPG
Image: 01302025PAAN0113.JPG
Image: 01302025PAAN0301.JPG
Image: 01302025PAAN0301C.JPG
Image: 01302025PAAN0302.JPG
Image: 01302025PAAN0302C.JPG
Image: 01302025PAAN0303.JPG
Image: 01302025PAAN0303C.JPG
Image: 01302025PAAN0304.JPG
Image: 01302025PAAN0304C.JPG
Image: 01302025PAAN0305.JPG
Image: 01302025PAAN0305C.JPG
Image: 01302025PAAN0306.JPG
Image: 01302025PAAN0306C.JPG
Image: 01302025PAAN0307.JPG
Image: 01302025PAAN0308.JPG
Image: 01302025PAAN0309.JPG
Image: 

In [25]:
# shark center points - convert to df
centerpoint_columns = [
    'filename', 'label', 'center_y', 'center_x', 'relative_altitude', 'gimbal_pitch_deg',
    'date_time', 'gps_lon', 'gps_lat', 'image_width', 'image_height', 'aircraft', 'site', 'flight']

centerpoint_df = pd.DataFrame(centerpoint_data, columns=centerpoint_columns)

In [27]:
# calibration points - convert to df
calibrationpoints_columns = [
    'filename', 'label', 'relative_altitude', 'gimbal_pitch_deg', 'date_time', 'gps_lon', 
    'gps_lat', 'image_width', 'image_height', 'endpoint1_x', 'endpoint1_y', 
    'endpoint2_x', 'endpoint2_y', 'aircraft', 'site', 'flight']

calibrationpoints_df = pd.DataFrame(calibrationpoints_data, columns=calibrationpoints_columns)

# compute calibration object length (pixels)
calibrationpoints_df['length_pixels'] = np.sqrt( 
    (calibrationpoints_df['endpoint2_x'] - calibrationpoints_df['endpoint1_x'])**2 +
    (calibrationpoints_df['endpoint2_y'] - calibrationpoints_df['endpoint1_y'])**2)

# drop unused columns
calib_len_trimmed = calib_len.drop(columns=['description']) # drop unused column
calibrationpoints_df_trimmed = calibrationpoints_df.drop(columns = ['endpoint1_x', 'endpoint1_y', 'endpoint2_x', 'endpoint2_y'])

# clean data
calibrationpoints_df_trimmed['label'] = calibrationpoints_df_trimmed['label'].str.strip() # strip whitespace
calib_len_trimmed['label'] = calib_len_trimmed['label'].str.strip() # strip whitespace

# merge with true calibration object lengths
calibration_df = calibrationpoints_df_trimmed.merge(calib_len_trimmed, on='label', how='left')

In [28]:
# write out data

# center points 
centerpoint_csv_file = os.path.join(xml_dir, 'centerpoints.csv')
centerpoint_df.to_csv(centerpoint_csv_file)

# calibration points
calibrationpoints_csv_file = os.path.join(xml_dir, 'calibrationpoints.csv')
calibration_df.to_csv(calibrationpoints_csv_file)